# lesson09 learning｜Selenium Notebook

先從上到下執行設定和函式定義格。每次示範或檢查會啟動練習站及 Chrome，結束時會關閉；請等待執行完成後再執行下一格。


In [ ]:

import os
import sys
import time
from pathlib import Path

import os
from pathlib import Path

# 從教材根目錄或本堂資料夾啟動 Notebook 都可以。
current = Path.cwd().resolve()
lesson_name = 'lesson09_selenium_basics'
candidates = [current, *current.parents]
lesson_path = next((p for p in candidates if p.name == lesson_name and (p / 'fixtures').is_dir()), None)
if lesson_path is None:
    lesson_path = next((p / 'lessons' / lesson_name for p in candidates if (p / 'lessons' / lesson_name / 'fixtures').is_dir()), None)
if lesson_path is None:
    raise FileNotFoundError('找不到本堂教材。請用 VS Code 開啟完整教材資料夾，重新啟動核心後再執行。')
os.chdir(lesson_path)
print('工作資料夾：', Path.cwd())
sys.path.insert(0, str(Path.cwd().parents[1] / "tools"))

import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

from practice_site import TEST_PASSWORD, TEST_USER, start_server, stop_server

HEADLESS = os.environ.get("HEADLESS") == "1"




## new_driver


In [ ]:
def new_driver():
    """建立一個 Chrome。Selenium 4 會自動準備對應版本的 chromedriver（第一次需要網路）。"""
    options = webdriver.ChromeOptions()
    if HEADLESS:
        options.add_argument("--headless=new")  # 不顯示視窗
    options.add_argument("--window-size=1280,900")
    return webdriver.Chrome(options=options)


## title


In [ ]:
def title(text):
    print(f"\n=== {text} ===")


## demo_01_preview


In [ ]:
def demo_01_preview(driver, base_url):
    title("01｜今天的成果預覽")
    driver.get(f"{base_url}/js-list")
    items = driver.find_elements(By.CSS_SELECTOR, "li.course")
    print("網頁標題：", driver.title)
    print("瀏覽器看到的項目數量：", len(items))
    print("第一個項目：", items[0].text)


## demo_02_requests_vs_selenium


In [ ]:
def demo_02_requests_vs_selenium(driver, base_url):
    title("02｜requests 看不到，Selenium 看得到")
    # 用 requests：只拿到原始 HTML，JavaScript 還沒執行
    raw_html = requests.get(f"{base_url}/js-list", timeout=10).text
    raw_items = BeautifulSoup(raw_html, "html.parser").select("li.course")
    print("requests + BeautifulSoup 找到：", len(raw_items), "個")

    # 用 Selenium：瀏覽器執行完 JavaScript 才取資料
    driver.get(f"{base_url}/js-list")
    print("Selenium 找到：", len(driver.find_elements(By.CSS_SELECTOR, "li.course")), "個")
    print("原因：原始 HTML 裡只有空的 <ul id=\"list\">，項目是 JavaScript 執行後才加進去的。")


## demo_03_start_and_quit


In [ ]:
def demo_03_start_and_quit():
    title("03｜啟動與關閉瀏覽器")
    # with 區塊結束時會自動關閉瀏覽器。沒關閉會留下殘留的 Chrome 與 chromedriver 程序。
    with new_driver() as driver:
        driver.get("about:blank")
        print("瀏覽器名稱：", driver.capabilities["browserName"])
        print("目前網址：", driver.current_url)
    print("with 區塊結束，瀏覽器已關閉")


## demo_04_find_elements


In [ ]:
def demo_04_find_elements(driver, base_url):
    title("04｜找元素：和 BeautifulSoup 幾乎一樣")
    driver.get(f"{base_url}/search")

    # find_element：找第一個，找不到會丟出 NoSuchElementException
    search_box = driver.find_element(By.ID, "q")
    print("搜尋框的 name 屬性：", search_box.get_attribute("name"))

    # find_elements：找全部，找不到回傳空 list（不會出錯）
    results = driver.find_elements(By.CSS_SELECTOR, "li.result")
    print("結果數量：", len(results), "｜第一筆文字：", results[0].text)

    links = driver.find_elements(By.CSS_SELECTOR, "nav a")
    print("導覽列連結：", [(link.text, link.get_attribute("href").replace(base_url, "")) for link in links])

    print("\n常用的 By：ID、CSS_SELECTOR、CLASS_NAME、TAG_NAME、LINK_TEXT（XPATH 下一堂）")
    print("CSS 選擇器的寫法和 BeautifulSoup 的 select 完全相同，直接沿用。")


## demo_05_page_source


In [ ]:
def demo_05_page_source(driver, base_url):
    title("05｜把渲染後的 HTML 交給 BeautifulSoup")
    driver.get(f"{base_url}/js-list")
    soup = BeautifulSoup(driver.page_source, "html.parser")
    courses = [item.get_text(strip=True) for item in soup.select("li.course")]
    print("用 BeautifulSoup 解析 driver.page_source：", courses)
    print("好處：前面學的解析、清理、pandas 都能直接接上。")

    saved = Path("fixtures/delayed_rendered.html").read_text(encoding="utf-8")
    print("\n存好的『瀏覽器渲染後』快照裡有 #result：", 'id="result"' in saved)


## demo_06_click


In [ ]:
def demo_06_click(driver, base_url):
    title("06｜點擊與導覽")
    driver.get(base_url + "/")
    driver.find_element(By.LINK_TEXT, "搜尋").click()
    print("點擊後的網址：", driver.current_url.replace(base_url, ""))
    driver.back()
    print("上一頁：", driver.current_url.replace(base_url, "") or "/")


## demo_07_search_form


In [ ]:
def demo_07_search_form(driver, base_url):
    title("07｜在表單輸入文字")
    driver.get(f"{base_url}/search")
    search_box = driver.find_element(By.ID, "q")
    search_box.clear()
    search_box.send_keys("python")

    # 送出表單：點擊按鈕
    driver.find_element(By.ID, "search-button").click()

    # 重點：點擊之後，新頁面可能還沒載入完，程式就已經往下執行了，會讀到「舊頁面」的內容。
    # 所以要明確等待：等網址出現 q=python（代表新頁面來了），再去讀結果。
    WebDriverWait(driver, 10).until(EC.url_contains("q=python"))

    count = driver.find_element(By.ID, "count").text
    titles = [item.text for item in driver.find_elements(By.CSS_SELECTOR, "li.result span.title")]
    print("網址：", driver.current_url.replace(base_url, ""))
    print("搜尋 python：", count)
    print("前兩筆：", titles[:2])


## demo_08_login


In [ ]:
def demo_08_login(driver, base_url):
    title("08｜登入、讀取會員頁、登出")
    driver.get(f"{base_url}/login")
    driver.find_element(By.ID, "username").send_keys(TEST_USER)
    driver.find_element(By.ID, "password").send_keys(TEST_PASSWORD)
    driver.find_element(By.ID, "login-button").click()

    # 換頁後用「等待」確認新頁面出現，比直接找元素更可靠
    welcome = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "welcome")))
    print("登入後：", welcome.text)

    rows = driver.find_elements(By.CSS_SELECTOR, "#orders tbody tr")
    print("訂單數量：", len(rows), "｜第一筆：", [cell.text for cell in rows[0].find_elements(By.TAG_NAME, "td")])

    driver.find_element(By.ID, "logout").click()
    print("登出後網址：", driver.current_url.replace(base_url, "") or "/")


## demo_09_waiting


In [ ]:
def demo_09_waiting(driver, base_url):
    title("09｜等待：動態內容需要時間")
    # 等待是 Selenium 最重要的觀念：頁面載入、換頁、JavaScript 產生內容，都需要時間。
    driver.get(f"{base_url}/delayed")
    started = time.time()

    try:
        driver.find_element(By.ID, "result")
    except NoSuchElementException:
        print("馬上找：找不到 #result（內容還沒出現）")

    # 明確等待：最多等 10 秒，條件成立就立刻往下走
    result = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "result")))
    print(f"等了 {time.time() - started:.1f} 秒後找到：", result.text)
    print("不要用固定的 time.sleep：等太短會失敗，等太長浪費時間。")


## demo_10_screenshots


In [ ]:
def demo_10_screenshots(driver, base_url):
    title("10｜截圖")
    Path("output").mkdir(exist_ok=True)

    driver.get(f"{base_url}/search?q=python")
    page_path = Path("output/search_python.png")
    driver.save_screenshot(str(page_path))

    element_path = Path("output/results_only.png")
    driver.find_element(By.ID, "results").screenshot(str(element_path))
    print("整頁截圖：", page_path, page_path.stat().st_size, "bytes")
    print("單一元素截圖：", element_path, element_path.stat().st_size, "bytes")


## demo_11_errors


In [ ]:
def demo_11_errors(driver, base_url):
    title("11｜常見的錯誤")
    driver.get(f"{base_url}/search")

    try:
        driver.find_element(By.ID, "no-such-id")
    except NoSuchElementException as error:
        print("NoSuchElementException：", str(error).splitlines()[0])

    try:
        WebDriverWait(driver, 1).until(EC.presence_of_element_located((By.ID, "no-such-id")))
    except TimeoutException:
        print("TimeoutException：等了 1 秒條件還是沒成立")

    print("\n遇到錯誤先想：")
    print("  1. 選擇器對嗎？（用 F12 確認）")
    print("  2. 元素出現了嗎？（要不要等待）")
    print("  3. 換頁後，舊的元素已經失效（StaleElementReferenceException），要重新找。")


## main


In [ ]:
def main():
    selected = sorted(sys.argv[1:] or list(DEMOS) + ["03"])
    server, base_url = start_server()
    print("本機練習站：", base_url, "｜無頭模式：", HEADLESS)
    driver = None
    try:
        for key in selected:
            if key == "03":
                demo_03_start_and_quit()
            elif key in DEMOS:
                if driver is None:
                    driver = new_driver()
                DEMOS[key](driver, base_url)
            else:
                print("沒有這個單元：", key)
    finally:
        if driver is not None:
            driver.quit()
        stop_server(server)
    print("\n全部完成。練習站與瀏覽器都已關閉。")


## 執行 demo_01_preview
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_01_preview(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_02_requests_vs_selenium
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_02_requests_vs_selenium(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_03_start_and_quit
本格結束後會關閉瀏覽器與練習站。


In [ ]:
demo_03_start_and_quit()


## 執行 demo_04_find_elements
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_04_find_elements(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_05_page_source
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_05_page_source(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_06_click
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_06_click(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_07_search_form
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_07_search_form(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_08_login
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_08_login(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_09_waiting
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_09_waiting(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_10_screenshots
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_10_screenshots(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)


## 執行 demo_11_errors
本格結束後會關閉瀏覽器與練習站。


In [ ]:
server, base_url = start_server()
driver = None
try:
    driver = new_driver()
    demo_11_errors(driver, base_url)
finally:
    if driver is not None:
        driver.quit()
    stop_server(server)
